# Test MaternKernelTorch

Quick validation of the new GPU-accelerated Matern kernel.

In [ ]:
import torch
import numpy as np
import time
import matplotlib.pyplot as plt

# Import both old and new implementations
from MaternKernel import matern_Kernel  # Original (NumPy/SciPy)
from MaternKernelTorch import MaternKernelTorch, matern_Kernel_torch  # New (PyTorch)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. Correctness Test: Compare with Original

In [ ]:
# Generate test data
np.random.seed(42)
X_np = np.random.randn(100, 15)
X_torch = torch.from_numpy(X_np).to(device)

print("Comparing kernel matrices:")
print("=" * 60)

for nu in [0.5, 1.5, 2.5, 5.0, 12.73]:
    # Original implementation
    kernel_old = matern_Kernel(X_np, nu=nu)
    K_old = kernel_old.compute()
    
    # New implementation
    kernel_new = MaternKernelTorch(nu=nu, device=device)
    K_new = kernel_new(X_torch).cpu().numpy()
    
    # Compare
    max_diff = np.abs(K_old - K_new).max()
    rel_diff = np.abs(K_old - K_new).mean() / np.abs(K_old).mean()
    
    status = "OK" if max_diff < 1e-6 else "CHECK"
    print(f"nu={nu:>5.2f}: max_diff={max_diff:.2e}, rel_diff={rel_diff:.2e} {status}")

## 2. Performance Benchmark: CPU vs GPU

In [ ]:
# Benchmark different sizes
sizes = [100, 500, 1000, 2000, 5000]
nu = 12.73  # Your use case!

results = []

for n in sizes:
    print(f"\nBenchmarking n={n}...")
    
    X_np = np.random.randn(n, 15)
    X_cpu = torch.from_numpy(X_np)
    X_gpu = X_cpu.to('cuda') if torch.cuda.is_available() else X_cpu
    
    # Original NumPy implementation
    start = time.time()
    kernel_old = matern_Kernel(X_np, nu=nu)
    K_old = kernel_old.compute()
    time_numpy = time.time() - start
    
    # PyTorch CPU
    kernel_cpu = MaternKernelTorch(nu=nu, device='cpu')
    start = time.time()
    K_cpu = kernel_cpu(X_cpu)
    time_torch_cpu = time.time() - start
    
    # PyTorch GPU (if available)
    if torch.cuda.is_available():
        kernel_gpu = MaternKernelTorch(nu=nu, device='cuda')
        torch.cuda.synchronize()
        start = time.time()
        K_gpu = kernel_gpu(X_gpu)
        torch.cuda.synchronize()
        time_torch_gpu = time.time() - start
    else:
        time_torch_gpu = float('nan')
    
    results.append({
        'n': n,
        'numpy': time_numpy,
        'torch_cpu': time_torch_cpu,
        'torch_gpu': time_torch_gpu
    })
    
    speedup = time_numpy / time_torch_gpu if not np.isnan(time_torch_gpu) else 0
    print(f"  NumPy: {time_numpy:.3f}s, PyTorch CPU: {time_torch_cpu:.3f}s, GPU: {time_torch_gpu:.3f}s")
    print(f"  GPU Speedup: {speedup:.1f}x")

In [ ]:
# Plot results
import pandas as pd
df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df['n'], df['numpy'], 'o-', label='NumPy (original)', linewidth=2, markersize=8)
ax.plot(df['n'], df['torch_cpu'], 's-', label='PyTorch CPU', linewidth=2, markersize=8)
if torch.cuda.is_available():
    ax.plot(df['n'], df['torch_gpu'], '^-', label='PyTorch GPU', linewidth=2, markersize=8)

ax.set_xlabel('Number of samples (n)', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title(f'Matern Kernel Computation Time (nu={nu})', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

## 3. Test Learnable Nu (For Future Extensions)

In [ ]:
# Test that gradients flow through nu
X = torch.randn(50, 15, dtype=torch.float64, device=device)

kernel = MaternKernelTorch(nu=5.0, learn_nu=True, device=device, dtype=torch.float64)
print(f"Initial nu: {kernel.nu.item():.4f}")

# Compute kernel and loss
K = kernel(X)
target_K = torch.eye(50, dtype=torch.float64, device=device)  # Dummy target
loss = ((K - target_K) ** 2).mean()

# Backward pass
loss.backward()

print(f"Loss: {loss.item():.6f}")
print(f"Gradient of nu: {kernel.nu.grad.item():.6e}")
print("\nGradients are flowing through nu!")

## 4. Test Backward Compatibility

In [ ]:
# Test drop-in replacement API
X_np = np.random.randn(100, 15)

# Old API
kernel_old = matern_Kernel(X_np, nu=12.73)
K_old = kernel_old.compute()
Y_old = kernel_old.sample(20)

# New API (same interface!)
kernel_new = matern_Kernel_torch(X_np, nu=12.73, device=device)
K_new = kernel_new.compute()
Y_new = kernel_new.sample(20)

print(f"Old kernel shape: {K_old.shape}")
print(f"New kernel shape: {K_new.shape}")
print(f"Old sample shape: {Y_old.shape}")
print(f"New sample shape: {Y_new.shape}")
print(f"Kernel max diff: {np.abs(K_old - K_new).max():.2e}")
print("\nBackward compatibility works!")

## 5. Full Compositional Data Generation Test

In [ ]:
from MaternKernelTorch import CompositionalDataGenerator

# Generate compositional data like in the paper
generator = CompositionalDataGenerator(
    d_input=15,
    d_intermediate=3,
    d_output=20,
    device=device
)

# Your paper's parameter sweep: nu_g and nu_h
nu_g = 2.0   # Smoothness of g (dimensionality reduction)
nu_h = 12.73  # Smoothness of h (your value!)

print(f"Generating compositional data with nu_g={nu_g}, nu_h={nu_h}...")
X, Y = generator.generate(nu_g=nu_g, nu_h=nu_h, n_samples=1000, seed=42)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"X device: {X.device}")
print(f"Y device: {Y.device}")

## Summary

The new `MaternKernelTorch` provides:

1. **GPU acceleration** for faster experiments
2. **Learnable nu** for future research extensions
3. **Backward compatibility** with your existing code
4. **Compositional data generation** built-in

To use in your experiments, just replace:
```python
from MaternKernel import matern_Kernel
```
with:
```python
from MaternKernelTorch import matern_Kernel_torch as matern_Kernel
```